In [ ]:
!pip install pandas matplotlib seaborn numpy


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# Set style for all plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (10, 6)

# Paths to CSVs (assuming notebook is inside /Report)
BASELINE_DIR = '../Results/baseline'
ESTIMATED_DIR = '../Results/estimated'
PLOT_DIR = 'plot'

# Load data
df_base_aicore2 = pd.read_csv(f'{BASELINE_DIR}/performance_baseline_results_devcloudaicore2.2.csv')
df_base_aicore4 = pd.read_csv(f'{BASELINE_DIR}/performance_baseline_results_devcloudaicore4.0.csv')
df_base_cubemx = pd.read_csv(f'{BASELINE_DIR}/performance_baseline_results_cubemx.csv')
df_base_aistudio = pd.read_csv(f'{BASELINE_DIR}/performance_baseline_results_aistudio.csv')

df_est_aicore2 = pd.read_csv(f'{ESTIMATED_DIR}/performance_estimated_results_devcloudaicore2.2.csv')
df_est_aicore4 = pd.read_csv(f'{ESTIMATED_DIR}/performance_estimated_results_devcloudaicore4.0.csv')

# Pre-processing: convert Inference Time to numeric (coerces 'FAIL' to NaN)
for df in [df_base_aicore2, df_base_aicore4, df_base_cubemx, df_base_aistudio]:
    df['Inference Time (ms)'] = pd.to_numeric(df['Inference Time (ms)'], errors='coerce')

df_base_aicore2.head()

# NPU Performance Analysis
This notebook analyzes the performance of the STM32N6 NPU by comparing actual execution data extracted from X-CUBE-AI and AI STUDIO compilers.
We explore execution times, memory usage, and architectural limits.

### 1. NPU Efficiency: The Impact of Operator Fusion

In [ ]:
# Comparison between standard conv2d and conv2d + maxpool
models_fusion = ['baseline_conv2d_16f_3x3_int8', 'baseline_conv2d_poll_int8', 'baseline_conv2d_pool_int8']
df_fusion = df_base_aicore2[df_base_aicore2['Model Name'].isin(models_fusion)].copy()

df_fusion['Layer'] = df_fusion['Model Name'].apply(lambda x: 'Conv2D + MaxPool' if 'poll' in x or 'pool' in x else 'Conv2D')

fig, ax1 = plt.subplots()
ax1.bar(df_fusion['Layer'], df_fusion['Inference Time (ms)'], color=['#3498db', '#e74c3c'], width=0.4)
ax1.set_ylabel('Inference Time (ms)', fontweight='bold')
ax1.set_title('Operator Fusion: Conv2D vs Conv2D + MaxPool\n(Time does not double thanks to NPU fusion)', fontweight='bold')

plt.savefig(f'{PLOT_DIR}/1_operator_fusion.png', bbox_inches='tight')
plt.show()

### 2. Efficiency by Kernel Type
Comparison between Standard, Pointwise (1x1), and Depthwise convolutions. We can see how the NPU manages different memory access patterns in parallel.

In [ ]:
models_type = ['baseline_conv2d_3x3_int8', 'baseline_conv2d_1x1_int8', 'baseline_depthwise_3x3_int8']
df_type = df_base_aicore2[df_base_aicore2['Model Name'].isin(models_type)].copy()

name_map = {
    'baseline_conv2d_3x3_int8': 'Conv2D (3x3)',
    'baseline_conv2d_1x1_int8': 'Pointwise (1x1)',
    'baseline_depthwise_3x3_int8': 'Depthwise (3x3)'
}
df_type['Type'] = df_type['Model Name'].map(name_map)

fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
width = 0.35
x = np.arange(len(df_type['Type']))

bar1 = ax1.bar(x - width/2, df_type['Inference Time (ms)'], width, label='Inference Time (ms)', color='#2ecc71')
bar2 = ax2.bar(x + width/2, df_type['MACC'], width, label='MACC', color='#9b59b6')

ax1.set_ylabel('Inference Time (ms)')
ax2.set_ylabel('MACC Operations')
ax1.set_xticks(x)
ax1.set_xticklabels(df_type['Type'])

fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.9))
plt.title('Efficiency by Convolution Type', fontweight='bold')

plt.savefig(f'{PLOT_DIR}/2_kernel_efficiency.png', bbox_inches='tight')
plt.show()

### 3. Computational Load vs Latency (Compute Bound vs Memory Bound)

In [ ]:
df_valid = df_base_aicore2.dropna(subset=['Inference Time (ms)'])

plt.figure(figsize=(10,6))
sns.scatterplot(data=df_valid, x='MACC', y='Inference Time (ms)', hue='Layer Type', s=200, palette='Set2')

for i in range(df_valid.shape[0]):
    plt.text(df_valid['MACC'].iloc[i], df_valid['Inference Time (ms)'].iloc[i] + 0.01, 
             df_valid['Model Name'].iloc[i].replace('baseline_','').replace('_int8',''), 
             fontsize=9)

plt.title('MACC vs Inference Time', fontweight='bold')
plt.xlabel('MACC (Operations)')
plt.ylabel('Inference Time (ms)')

plt.savefig(f'{PLOT_DIR}/3_macc_vs_latency.png', bbox_inches='tight')
plt.show()

### 4. Architectural Limits: RAM Saturation
We show what happens when the model exceeds the available static SRAM memory (approx 1.5MB). The 32f_7x7 model saturates memory leading to HW Timeout.

In [ ]:
models_limit = ['baseline_conv2d_32f_5x5_int8', 'baseline_conv2d_32f_7x7_int8', 'baseline_conv2d_3x3_int8']
df_limit = df_base_aicore2[df_base_aicore2['Model Name'].isin(models_limit)].copy()

df_limit['Kernel'] = df_limit['Kernel Size']

plt.figure(figsize=(8,6))
sns.barplot(data=df_limit.sort_values('RAM Usage (KiB)'), x='Kernel', y='RAM Usage (KiB)', palette='magma')
plt.axhline(y=1500, color='r', linestyle='--', label='SRAM Limit (~1.5MB)')

plt.title('RAM Growth and Saturation (Architectural Limit)', fontweight='bold')
plt.legend()

plt.savefig(f'{PLOT_DIR}/4_ram_saturation.png', bbox_inches='tight')
plt.show()

### 5. Compiler Comparison: AICore v2.2 vs AICore v4.0

In [ ]:
df_compare = pd.merge(df_base_aicore2[['Model Name', 'Inference Time (ms)']], 
                      df_base_aicore4[['Model Name', 'Inference Time (ms)']], 
                      on='Model Name', suffixes=('_v2.2', '_v4.0'))

df_compare = df_compare.dropna()

df_compare.set_index('Model Name').plot(kind='bar', figsize=(12,6), color=['#f39c12', '#8e44ad'])
plt.title('Inference Time Comparison: AICore v2.2 vs v4.0', fontweight='bold')
plt.ylabel('Inference Time (ms)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

plt.savefig(f'{PLOT_DIR}/5_compiler_comparison.png')
plt.show()